---
title: "Chapter 33: Predicting household spend"
---

In November 2019, a single-person household in Alabama, reporting an income between $25,000 and $49,999, bought a 65-inch Samsung 4K television, a PlayStation 4 Pro, a Microsoft Surface tablet, an Apple Watch, a GoPro camera bundle, a trampoline, and a kayak. Fifty-eight items, spread across four days. The total came to $11,709.16.

For the three months before that, the same household had spent an average of $49.47 a month. Across the roughly four years this account appears in the data, its lifetime total is $16,237.77. One month accounts for 72 percent of everything it has ever bought on the platform.

Nothing in this household's survey answers, age bracket, income, education, state, explains it. There's no flag for it and no obvious trigger. It's one row among 1.85 million, sitting in the data whether or not anyone thinks to look for it.

Chapters 28 through 32 built the machinery for exactly this kind of question: how to turn a vague business problem into a five-component specification, how the nine-stage workflow moves data through a model and into production, how scikit-learn's Transformer and Predictor interfaces work, how a Pipeline strings them together without leaking information across a split. All of it ran on hotel booking data that was already clean enough to teach with.

This chapter runs the same machinery on a dataset nobody cleaned for teaching: five years of real Amazon purchase histories from 5,027 U.S. households, linked to a demographic and lifestyle survey each household completed once. The household above is what "real" looks like. It is also, as it turns out, not the point of this chapter, just proof that the data has teeth.

## 1. Problem framing

Here is how the request actually arrived. A retention team at a company with access to purchase-and-demographics data like this, for a panel of customers, asks a simple question:

"We want to know which customers are worth investing more in."

That's the whole brief. No target variable, no metric, no mention of a model. This is what a real ask sounds like, and figuring out what it actually means is the first and most consequential part of the job, not a formality to clear before the "real" work starts.

### What does "worth investing in" even mean?

Before opening a single file, the phrase needs to be pulled apart, because it collapses at least four different problems into one sentence, and they are not interchangeable:

- Customers who already spend the most. A ranking of the past, nothing predictive about it.
- Customers likely to spend more going forward. A forecast.
- Customers likely to leave if nobody intervenes. A churn problem.
- Customers who look like existing high spenders but haven't been identified yet. A segmentation problem.

Each of these is a legitimate, well-posed data science problem. Each one produces a different target variable, a different model, and a different action the business takes on the output. Picking one without asking which is exactly how a team spends six months building the right system for the wrong question.

A follow-up question to the retention team narrows it down: "worth investing in" turns out to mean deciding who gets priority for a loyalty program with a fixed, limited budget. They want to rank customers by how much they are likely to spend going forward, using the demographic and behavioural data already on file, before that spending happens, not a scoreboard of who has spent the most so far.

That single answer resolves the first fork. It's a forecast of future value, not a description of past value, which already rules out simply sorting by historical total. It also starts pinning down Chapter 30's five-component spec, one piece at a time, in the order the questions actually forced them out:

**Target variable.** Not "past spend", predicted future spend per household. The next question is over what window, a month, a year, a lifetime, and the answer depends on how often the loyalty program actually reallocates its budget. If the business revisits the list annually, the target is a household's spend over the coming year, not the coming week.

**Task type and paradigm.** A number to predict, from labelled historical examples where the outcome is already known, is regression, supervised.

**Success metric.** "Worth investing in" is a ranking decision, not a per-household prediction the business will read one row at a time, so the metric that matters is how well the model orders households relative to each other, not how close each individual prediction lands. Mean absolute error still matters as a sanity check, but the metric the loyalty program actually cares about is whether the top-ranked decile really does outspend the rest.

**Acceptable threshold.** A model that can't beat "assume everyone spends the average" is not worth deploying. The concrete bar: households the model ranks in its top decile of predicted spend must, on average, outspend a randomly chosen household by a margin large enough to justify the loyalty program's cost per member, a number the business, not the data scientist, has to supply.

**Data specification.** Demographic and lifestyle survey answers, collected once per household, and purchase history spanning multiple years, at the individual-transaction grain, for 5,027 households.

<div class='ark-activity'>
<span class='ark-activity-title'><i class="bi bi-puzzle-fill"></i> Activity: write the spec card before reading further</span><br><br>
Using the same five components from Chapter 30 (target variable, task type and paradigm, success metric, acceptable threshold, data specification), write your own answer for this problem before looking at the resolved version below. Compare where you landed. If your target variable differs, that's fine, it's the kind of decision Chapter 30's Pro Tip already warned about: agree it with the stakeholder who will actually use the model, because a model optimised toward the wrong objective will hit that objective and still miss the point.
</div>

### The resolved spec card

| Component | Specification |
|---|---|
| Target variable | Household's total spend over the coming 12 months |
| Task type and paradigm | Regression, supervised |
| Success metric | Rank quality (does the top decile of predicted spend actually outspend the rest), MAE as a secondary check |
| Acceptable threshold | Top-decile households must outspend a randomly chosen household by a margin the loyalty program's budget can justify |
| Data specification | One-time demographic and lifestyle survey plus multi-year, transaction-level purchase history, 5,027 U.S. households |

That's the spec. It took five questions to get here, not a target variable declared in the first paragraph, and every one of those questions is a version of something a real stakeholder conversation would actually raise.

Next: turning this into something durable enough that the rest of the project can't quietly drift from it, a Pydantic object, not a paragraph anyone can reinterpret three sections from now.